# ALiBi：用距离直接调整 Attention Score

## 一、为什么提出 ALiBi？

前面学习的位置编码，都让 Attention 能够利用位置信息。但还有一个实际问题：**训练时使用较短的序列，推理时能否有效处理更长的序列？** 这称为**长度外推（Length Extrapolation）**。由于长序列训练的注意力计算和显存开销较高，研究者希望不必始终用很长的序列训练。

不同位置编码在这里有不同的限制。例如，**Learned Absolute PE** 如果只为训练长度建立位置表，推理时超出表长就无法直接查到对应位置；**Sinusoidal PE** 可以计算新位置，却不能因此保证模型在长序列上效果良好。

再看刚学过的 **T5**：它把相对距离分桶，再通过 `Embedding` 学习每个桶的位置偏置。远距离会逐渐合并；超过设定范围后，不同的远距离可能得到同一个偏置。**T5 仍然可以处理更长输入**，只是这个位置偏置不再区分那些距离。

那么，是否可以不使用有限的桶和位置参数表，而是让**任意距离都能直接算出位置偏置**？ALiBi 就采用了这个思路，并在语言模型上研究它的长度外推效果。

## 二、ALiBi 究竟改了什么？

先回忆 T5：内容分数 $c_{ij}$ 算出来以后，还要查表得到位置偏置，再把两者相加。

$$
\text{T5:}\qquad s_{ij}=c_{ij}+b_{\operatorname{bucket}(D_{ij})},\qquad c_{ij}=\frac{q_i^\top k_j}{\sqrt{d_k}}.
$$

ALiBi **保留“在 Softmax 前给分数加位置偏置”的做法**，只是把“分桶、查表”换成了一个简单的距离惩罚。仍采用前面统一的距离定义 $D_{ij}=i-j$：Query 在位置 $i$，Key 在位置 $j$；在因果 Attention 中，允许关注的 Key 满足 $j\le i$，因此 $D_{ij}\ge0$。

$$
\boxed{s_{ij}=c_{ij}-mD_{ij}},\qquad m>0.
$$

为什么用减号？因为我们想给 Attention **一种近距离优先的倾向**，而不是规定近距离内容一定更重要。看一个 Query，假设四个 Key 的**内容分数都相同**，取 $m=0.5$：

| 按距离排列的展示编号 | Key 0（最近） | Key 1 | Key 2 | Key 3（最远） |
|---|---:|---:|---:|---:|
| 距离 $D$ | 0 | 1 | 2 | 3 |
| 位置偏置 $-0.5D$ | 0 | -0.5 | -1.0 | -1.5 |
| Softmax 权重（约） | **0.455** | 0.276 | 0.167 | 0.102 |

如果不加偏置，四个权重都是 $0.25$。加入负的距离偏置后，内容相同时，近处的权重更大；**远处若有更高的内容分数，仍然可以获得更大权重**。

> 这里的 Key 0～3 是按“从近到远”重新排列的展示编号。真实 Self-Attention 矩阵仍按 Token 原始位置排列，不要混淆。

## 三、为什么用线性，而不是非线性？

**非线性也可以。** 如果只是希望距离越远、惩罚越大，平方、对数等函数都能实现。ALiBi 选择线性，一个容易理解的性质要从 **Softmax** 看。

先回到前面已经得到的分数 $s_{ij}=c_{ij}-mD_{ij}$。**这还不是 Attention 权重**，需要经过 Softmax。对于同一个 Query $i$，Softmax 会将它对所有允许关注的 Key 的分数转换为权重：

$$
a_{ij}=\frac{e^{s_{ij}}}{\sum_{k\le i}e^{s_{ik}}}=\frac{e^{c_{ij}-mD_{ij}}}{\sum_{k\le i}e^{c_{ik}-mD_{ik}}}.
$$

**为什么说位置偏置会影响 Softmax？** 因为指数运算有 $e^{a-b}=e^a\cdot e^{-b}$，所以每个 Key 在归一化之前的分数可以拆成两部分：

$$
e^{c_{ij}-mD_{ij}}=\underbrace{e^{c_{ij}}}_{\text{内容匹配}}\cdot\underbrace{e^{-mD_{ij}}}_{\text{距离因子}}.
$$

也就是说，**内容匹配越高，原本得到的分数越大；距离越远，$e^{-mD_{ij}}$ 越小，会把这个分数向下压。** 这里的 $e^{-mD_{ij}}$ 只是 Softmax *归一化之前*的一个乘数，**并不是最终的 Attention 权重**，因为所有 Key 的结果还需要共同除以分母。

为了单独观察距离的影响，假设这个 Query 对所有 Key 的内容分数都相同（$c_{ij}=c$）。此时共同的 $e^c$ 会在分子、分母中约掉：

$$
a_{ij}=\frac{e^{-mD_{ij}}}{\sum_{k\le i}e^{-mD_{ik}}}.
$$

现在就能看出线性距离惩罚的特点了。若 $m=0.5$，距离为 $0$、$1$ 的两个 Key，其距离因子分别是 $1$、$e^{-0.5}\approx0.607$；**远一个位置，未归一化的权重因子约变为原来的 $0.607$ 倍**。由于它们属于同一行 Softmax，分母相同，所以最终权重之比也是 $0.607$：

$$
\frac{a(d+1)}{a(d)}=\frac{e^{-m(d+1)}}{e^{-md}}=e^{-m}.
$$

因此，**ALiBi 在 Score 上采用线性惩罚，经过 Softmax 后，会让内容相同的 Key 随距离以固定比例衰减**。若使用平方惩罚，每增加一个位置带来的额外惩罚会随距离增大；若使用对数惩罚，远距离惩罚则增加得更慢。

线性并不是数学上唯一正确的选择；它提供了简单、稳定的距离规则。ALiBi 作者也试过其他惩罚形式，最终采用了实验效果较好的线性形式。

## 四、多头 Attention 为什么使用不同斜率？

刚才只考虑一个注意力头，所有距离由同一个 $m$ 控制。如果 $m$ 较大，远距离受到的惩罚更强；如果 $m$ 较小，远距离受到的惩罚更弱。

多头 Attention 原本就希望不同 Head 学到不同的信息，因此 ALiBi 为**每个 Head 固定一个不同的斜率** $m_h$，使各 Head 具有不同强度的距离偏好。斜率由预设规则产生，**原始 ALiBi 不将其作为可训练参数**。

最终公式只是把刚才的 $m$ 换成 $m_h$：

$$
\boxed{s_{ij}^{(h)}=\frac{(q_i^{(h)})^\top k_j^{(h)}}{\sqrt{d_h}}-m_h(i-j)},\qquad j\le i.
$$

在因果 Attention 中，**未来位置仍然需要 Mask**；ALiBi 的距离惩罚不能替代它。之后才对分数进行 Softmax。

最后回到长度外推：无论距离是 20、128 还是 512，都可以直接计算 $-m_hD$，**无需扩展位置表，也不会把所有超远距离合并为同一个偏置**。但“可以计算更长距离”不等于“必然能有效利用任意长的上下文”，实际效果仍需要实验验证。

**记住这一个区别即可：** T5 是“**距离 → 分桶 → 查可学习偏置**”；ALiBi 是“**距离 × 固定斜率 → 直接得到偏置**”。

参考：Press, Smith & Lewis, [*Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation*](https://arxiv.org/abs/2108.12409), ICLR 2022；[官方实现](https://github.com/ofirpress/attention_with_linear_biases)。